# Lab 4 — Asistente de compras por voz (MVP reproducible)

Notebook para Colab. Flujo: ESCI → catálogo → búsqueda RAG → agente con herramientas → tienda simulada → voz. El agente usa código explícito para operaciones y validaciones; el LLM redacta respuestas apoyadas en contexto recuperado.

**Requisitos cubiertos:** corpus ESCI en español, consultas agrupadas por `query_id`, split train/test, índice vectorial, baseline léxico, nDCG@10, tienda sintética con semilla, estado conversacional, carrito/cotización/pedido idempotente, ASR/TTS local, escenarios y registro. No usa APIs externas de inferencia.

> Ejecutar en Colab con GPU. Las celdas de voz descargan modelos preentrenados dentro del runtime. La referencia de voz debe ser propia o autorizada.


## 1. Entorno y configuración

In [ ]:
!pip -q install pandas pyarrow sentence-transformers faiss-cpu scikit-learn transformers accelerate


In [ ]:
import os, random, json, time, re, uuid, shutil
from pathlib import Path
import numpy as np
import pandas as pd

SEED = 42
random.seed(SEED); np.random.seed(SEED)
DATA_DIR = Path('/content/data'); DATA_DIR.mkdir(exist_ok=True)
ARTIFACTS = Path('/content/artifacts'); ARTIFACTS.mkdir(exist_ok=True)


## 2. Cargar y explorar ESCI
Sube los dos Parquet descargados desde el repositorio oficial a `data/`. `products` es catálogo; `examples` contiene pares consulta–producto y sus juicios de relevancia.

In [ ]:
from google.colab import files
# Ejecuta y selecciona shopping_queries_dataset_products.parquet y
# shopping_queries_dataset_examples.parquet
uploaded = files.upload()
for name, data in uploaded.items():
    (DATA_DIR / name).write_bytes(data)

products_path = DATA_DIR / 'shopping_queries_dataset_products.parquet'
examples_path = DATA_DIR / 'shopping_queries_dataset_examples.parquet'
products = pd.read_parquet(products_path)
examples = pd.read_parquet(examples_path)
print(products.shape, products.columns.tolist())
print(examples.shape, examples.columns.tolist())
print(products.product_locale.value_counts(dropna=False))
print(examples.esci_label.value_counts(dropna=False))
display(products.head(3)); display(examples.head(3))


### Interpretación de los datos
Una fila de `examples` significa “para esta consulta, este producto fue uno de los candidatos y recibió esta etiqueta”. `query_id` agrupa candidatos de la misma consulta. `product_id` conecta con la ficha en `products`. Etiquetas: E exacto, S sustituto, C complemento, I irrelevante. **No se incluirán etiquetas en el índice ni en el contexto del asistente.**

In [ ]:
# Explorar un ejemplo completo y comprobar el join
locale_vals = products.product_locale.dropna().unique().tolist()
print('Locales:', locale_vals)
# Ajusta el locale tras inspeccionar el resultado; en ESCI suele ser 'es_ES'.
LOCALE = next((x for x in locale_vals if str(x).lower().startswith('es')), locale_vals[0])
products_es = products[products.product_locale == LOCALE].copy()
examples_es = examples[examples.product_locale == LOCALE].copy()
print('Locale elegido:', LOCALE, '| productos:', len(products_es), '| ejemplos:', len(examples_es))
qid = examples_es.query_id.iloc[0]
joined = examples_es[examples_es.query_id == qid].merge(products_es, on=['product_locale','product_id'], how='left')
display(joined[['query','product_id','esci_label','product_title','product_brand']].head(10))


## 3. Preparar catálogo y datos de evaluación
Para mantener el notebook liviano, indexaremos hasta 5 000 productos en español. Separamos consultas por `query_id`; primero usamos train para desarrollo y test queda reservado para evaluación final.

In [ ]:
MAX_PRODUCTS = 5000
products_es = products_es.drop_duplicates(['product_locale','product_id']).head(MAX_PRODUCTS).reset_index(drop=True)
TEXT_COLS = ['product_title','product_description','product_bullet_point','product_brand','product_color']
for c in TEXT_COLS:
    if c not in products_es: products_es[c] = ''
    products_es[c] = products_es[c].fillna('').astype(str)
products_es['document'] = products_es.apply(lambda r: '\n'.join(f'{c}: {r[c]}' for c in TEXT_COLS if r[c].strip()), axis=1)
# Evaluación solo para pares cuyo producto está dentro del catálogo indexado
examples_es = examples_es[examples_es.product_id.isin(products_es.product_id)].copy()
train = examples_es[examples_es.split == 'train'].copy()
test = examples_es[examples_es.split == 'test'].copy()
print('Productos indexables:', len(products_es), 'query_id train:', train.query_id.nunique(), 'test:', test.query_id.nunique())


## 4. RAG básico: embeddings e índice FAISS
Se usa un modelo multilingüe de embeddings. El mismo modelo transforma productos y consultas. FAISS almacena vectores normalizados y devuelve los productos más similares.

In [ ]:
from sentence_transformers import SentenceTransformer
import faiss

EMBED_MODEL = 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'
embedder = SentenceTransformer(EMBED_MODEL)
texts = products_es.document.tolist()
embeddings = embedder.encode(texts, batch_size=64, show_progress_bar=True, normalize_embeddings=True).astype('float32')
index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)
print('Vectores indexados:', index.ntotal)


In [ ]:
def retrieve(query, k=10):
    qv = embedder.encode([query], normalize_embeddings=True).astype('float32')
    scores, ids = index.search(qv, min(k, len(products_es)))
    out = products_es.iloc[ids[0]].copy()
    out['score'] = scores[0]
    return out

display(retrieve('auriculares para escuchar música', 5)[['product_id','product_title','score']])


## 5. Baseline léxico y evaluación nDCG@10
Usamos TF-IDF como baseline léxico. La métrica usa juicios ESCI solo en esta celda de evaluación, nunca en `retrieve()` ni en el prompt del asistente.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel
from sklearn.preprocessing import normalize

vectorizer = TfidfVectorizer(max_features=100000, ngram_range=(1,2), min_df=2)
tfidf_products = vectorizer.fit_transform(products_es.document)
product_ids = products_es.product_id.tolist()
id_to_row = {pid:i for i,pid in enumerate(product_ids)}
def lexical_retrieve(query, k=10):
    qv = vectorizer.transform([query]); sims = linear_kernel(qv, tfidf_products).ravel()
    inds = np.argsort(sims)[::-1][:k]
    return [(product_ids[i], float(sims[i])) for i in inds]

REL = {'E':1.0, 'S':0.1, 'C':0.01, 'I':0.0}
def ndcg_at_k(ranked_ids, judged, k=10):
    gains = [REL.get(judged.get(pid,'I'),0.0) for pid in ranked_ids[:k]]
    dcg = sum((2**g-1)/np.log2(i+2) for i,g in enumerate(gains))
    ideal = sorted([REL.get(v,0.0) for v in judged.values()], reverse=True)[:k]
    idcg = sum((2**g-1)/np.log2(i+2) for i,g in enumerate(ideal))
    return dcg/idcg if idcg else 0.0

def eval_retriever(frame, mode='dense', limit=200):
    qids = frame.query_id.drop_duplicates().head(limit)
    vals=[]
    for qid in qids:
        g=frame[frame.query_id==qid]
        query=g.query.iloc[0]
        judged=dict(zip(g.product_id, g.esci_label))
        if mode=='dense': ranked=retrieve(query,10).product_id.tolist()
        else: ranked=[pid for pid,_ in lexical_retrieve(query,10)]
        vals.append(ndcg_at_k(ranked, judged,10))
    return float(np.mean(vals)), len(vals)

# Desarrollo con train; evaluación final ejecutar con test una sola vez.
print('Train dense:', eval_retriever(train,'dense'))
print('Train lexical:', eval_retriever(train,'lexical'))


## 6. Tienda sintética reproducible y herramientas
Los precios y existencias no vienen de ESCI: los generamos con semilla fija. Las funciones validan cantidades, presupuesto y stock. Confirmar vuelve a validar y usa `confirmation_id` para no duplicar pedidos.

In [ ]:
def restore_store(seed=SEED):
    rng=np.random.default_rng(seed)
    store=products_es[['product_id','product_title','product_brand']].copy()
    store['price']=np.round(rng.uniform(8,250, len(store)),2)
    store['stock']=rng.integers(0,21,len(store))
    return store.set_index('product_id').to_dict('index')

store=restore_store()
cart={}
orders={}
confirmations={}
def product_search(query,k=5): return retrieve(query,k)
def get_product(pid): return store.get(pid)
def set_quantity(pid, qty):
    if pid not in store: return {'ok':False,'message':'Producto no encontrado'}
    if not isinstance(qty,int) or qty<=0: return {'ok':False,'message':'Cantidad debe ser entero positivo'}
    if qty>store[pid]['stock']: return {'ok':False,'message':f"Stock insuficiente: quedan {store[pid]['stock']}"}
    cart[pid]=qty; return {'ok':True,'message':'Carrito actualizado'}
def quote(budget=None):
    lines=[{'product_id':p,'title':store[p]['product_title'],'qty':q,'price':store[p]['price'],'subtotal':round(q*store[p]['price'],2)} for p,q in cart.items()]
    total=round(sum(x['subtotal'] for x in lines),2)
    return {'lines':lines,'total':total,'within_budget': budget is None or total<=budget}
def confirm_order(confirmation_id, budget=None):
    if confirmation_id in confirmations: return confirmations[confirmation_id]
    q=quote(budget)
    if not q['lines']: return {'ok':False,'message':'Carrito vacío'}
    if not q['within_budget']: return {'ok':False,'message':'La cotización supera el presupuesto'}
    for pid,qty in cart.items():
        if store[pid]['stock']<qty: return {'ok':False,'message':f"Stock cambió para {pid}; no se registró el pedido"}
    # Actualización agrupada tras validar todas las líneas
    order_id=str(uuid.uuid4())
    for pid,qty in cart.items(): store[pid]['stock']-=qty
    result={'ok':True,'order_id':order_id,'quote':q}
    orders[order_id]=result; confirmations[confirmation_id]=result
    return result


## 7. Modelo local para redactar respuestas con contexto recuperado

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
LLM_ID='Qwen/Qwen2.5-1.5B-Instruct'
tokenizer=AutoTokenizer.from_pretrained(LLM_ID)
llm=AutoModelForCausalLM.from_pretrained(LLM_ID, torch_dtype='auto', device_map='auto')
def answer_with_rag(question, docs):
    context='\n'.join(f"- {r.product_title}. {r.product_description or ''} Marca: {r.product_brand or 'desconocida'}. Color: {r.product_color or 'desconocido'}. ID: {r.product_id}" for r in docs.itertuples())
    messages=[{'role':'system','content':'Responde en español usando solo el contexto. Si falta un dato, dilo. No inventes precios ni stock.'},{'role':'user','content':f'Contexto:\n{context}\n\nPregunta: {question}'}]
    prompt=tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs=tokenizer(prompt, return_tensors='pt').to(llm.device)
    output=llm.generate(**inputs, max_new_tokens=180, do_sample=False)
    return tokenizer.decode(output[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)


## 8. Agente mínimo con herramientas explícitas y estado conversacional
El modelo redacta; el control de operaciones queda en código. Esta versión sencilla admite búsqueda, añadir producto por ID, cotizar y confirmar; si la petición no indica producto/cantidad, solicita aclaración. No delega decisiones de inventario al LLM.

In [ ]:
conversation={'preferences':{},'budget':None,'last_results':None,'transcript':[]}
def assistant_turn(text, confirm=False):
    conversation['transcript'].append({'role':'user','text':text})
    low=text.lower()
    # Extraer presupuesto de forma básica
    m=re.search(r'(?:presupuesto|máximo|hasta)\s*(?:de)?\s*(\d+(?:[.,]\d+)?)',low)
    if m: conversation['budget']=float(m.group(1).replace(',','.'))
    if any(w in low for w in ['buscar','busco','recomienda','quiero','auricular','celular','zapatilla','producto']):
        res=product_search(text,5); conversation['last_results']=res
        response=answer_with_rag(text,res)
    elif 'cotiza' in low or 'cotización' in low:
        response=json.dumps(quote(conversation['budget']),ensure_ascii=False)
    elif confirm:
        response=json.dumps(confirm_order(str(uuid.uuid4()),conversation['budget']),ensure_ascii=False)
    else:
        response='¿Qué producto buscas y qué cantidad necesitas? Puedes indicar también un presupuesto.'
    conversation['transcript'].append({'role':'assistant','text':response})
    return response


## 9. Ablación sencilla
Ablación: comparar recuperación densa con recuperación léxica, sobre el mismo conjunto de consultas. Reportamos nDCG@10. No se ajustan parámetros usando test.

In [ ]:
# Ejecutar al final y una sola vez para test
print('Test dense:', eval_retriever(test,'dense',limit=200))
print('Test lexical:', eval_retriever(test,'lexical',limit=200))


## 10. ASR en español y prueba de corrección de transcripción

In [ ]:
!pip -q install faster-whisper
from faster_whisper import WhisperModel
asr=WhisperModel('small', device='cuda' if torch.cuda.is_available() else 'cpu', compute_type='float16' if torch.cuda.is_available() else 'int8')
def transcribe(path):
    segments,info=asr.transcribe(path, language='es', vad_filter=True)
    return ' '.join(s.text.strip() for s in segments)
# En Colab: subir audio, transcribir, mostrar texto y pedir revisión manual antes de assistant_turn().


## 11. Cargar perfil de voz preparado
El notebook `01_preparar_perfil_de_voz.ipynb` calcula y guarda un perfil XTTS en Google Drive. Aquí solo se carga ese perfil para generar audio; no se vuelve a procesar la grabación de referencia. No subas el perfil ni el audio al repositorio.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip -q install TTS
import torch
from TTS.api import TTS

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
tts = TTS('tts_models/multilingual/multi-dataset/xtts_v2').to(DEVICE)
VOICE_PROFILE = '/content/drive/MyDrive/lab4/voice_profile_winz.pt'
profile = torch.load(VOICE_PROFILE, map_location=DEVICE, weights_only=True)
gpt_cond_latent = profile['gpt_cond_latent'].to(DEVICE)
speaker_embedding = profile['speaker_embedding'].to(DEVICE)

def speak(text, out='/content/respuesta.wav'):
    result = tts.synthesizer.tts_model.inference(
        text=text, language='es', gpt_cond_latent=gpt_cond_latent,
        speaker_embedding=speaker_embedding, enable_text_splitting=True
    )
    import soundfile as sf
    sf.write(out, result['wav'], 24000)
    return out


## 12. Escenarios reproducibles y registro
El laboratorio solicita al menos 20 escenarios operativos, seis con audio y varias intervenciones en al menos tres. La tabla permite registrar resultado esperado/obtenido y evidencias. Ejecutar pruebas después de restaurar tienda y conversación en cada caso.

In [ ]:
SCENARIOS=[
 {'id':'S01','type':'texto','case':'Compra dentro de restricciones','expected':'Pedido confirmado'},
 {'id':'S02','type':'texto','case':'Consulta ambigua','expected':'Solicita aclaración'},
 {'id':'S03','type':'texto','case':'Cambio de cantidad','expected':'Conserva resto de preferencias'},
 {'id':'S04','type':'texto','case':'Producto agotado','expected':'Ofrece alternativa o informa'},
 {'id':'S05','type':'audio','case':'Transcripción corregible','expected':'Pide confirmar texto antes de comprar'},
 {'id':'S06','type':'audio','case':'Stock cambia y confirmación repetida','expected':'No pedido parcial ni duplicado'},
]
# Completar hasta 20 escenarios con variantes; registrar las conversaciones completas en JSONL.
def reset_session():
    global store,cart,orders,confirmations,conversation
    store=restore_store(SEED); cart={}; orders={}; confirmations={}
    conversation={'preferences':{},'budget':None,'last_results':None,'transcript':[]}

def save_record(record, path=ARTIFACTS/'scenario_log.jsonl'):
    with open(path,'a',encoding='utf-8') as f: f.write(json.dumps(record,ensure_ascii=False)+'\n')


## 13. Métricas y resultados a reportar
Completar la matriz de 20 escenarios; analizar cinco aciertos y cinco fallos. Para voz guardar audio, transcripción original/corregida, llamadas a herramientas, respuesta y estado final. Medir tiempos de carga inicial y de consulta por separado.

In [ ]:
results={
 'retrieval_ndcg10_dense': None,
 'retrieval_ndcg10_lexical': None,
 'scenario_success_rate': None,
 'restriction_violations': None,
 'asr_errors_quantities_amounts_products': None,
 'latency_by_stage_seconds': {},
 'voice_intelligibility_notes': None,
 'voice_similarity_notes': None,
}
print(json.dumps(results,indent=2,ensure_ascii=False))


## Límites conocidos del MVP
- Se indexa un máximo de 5 000 productos, seleccionados por orden del archivo; documentar este criterio o muestrear con semilla.
- La extracción de intención/presupuesto es básica; ampliar únicamente con casos de prueba que lo requieran.
- Para robustez de producción, las transacciones de tienda deben estar en BD transaccional; aquí se simulan en memoria con restauración determinista.
- XTTS puede requerir ajuste de versión en Colab y referencia de voz. Si falla, conservar el requisito y documentar limitación; no sustituir por API externa.
